In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk
from styles import *

from ipynb.fs.full.data_manager import initialise_file, save_user, validate_login, save_job, get_employer_jobs, get_job_applicants

from ipynb.fs.full.job_browsing import EmployeeDashboard

initialise_file()

root = tk.Tk()
root.title("WorkLink - Integrated Employment Exchange Workspace")
root.geometry("850x700")
root.config(bg=BACKGROUND_COLOR)

current_user_id = None
current_user_name = None
current_user_role = None

def clear_screen():
    """Wipes all nested elements from the root canvas grid layout frame securely prior to redraw mutations."""
    for widget in root.winfo_children():
        widget.destroy()

def show_role_selection():
    clear_screen()
    root.geometry("520x520")
    
    title = tk.Label(root, text="Welcome to WorkLink", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=40)
    
    subtitle = tk.Label(root, text="Please specify your ecosystem path access tier gateway:", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT)
    subtitle.pack(pady=10)

    emp_btn = tk.Button(root, text="I am an Corporate Employer", **BUTTON_STYLE, width=28, height=2, command=lambda: show_login("Employer"))
    emp_btn.pack(pady=15)

    seeker_btn = tk.Button(root, text="I am a Registered Job Seeker", **BUTTON_STYLE, width=28, height=2, command=lambda: show_login("Job Seeker"))
    seeker_btn.pack(pady=15)

    exit_btn = tk.Button(root, text="Close Terminal Frame Connection", **BUTTON_STYLE, bg=ERROR_COLOR, activebackground="#D62828", width=15, command=root.destroy)
    exit_btn.pack(pady=35)

def show_login(role):
    clear_screen()
    root.geometry("520x550")

    title = tk.Label(root, text=f"{role} Access Portal Link", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=30)

    tk.Label(root, text="Registered Identity Account Email Address", **LABEL_STYLE).pack(anchor="w", padx=100)
    email_entry = tk.Entry(root, width=35, **ENTRY_STYLE)
    email_entry.pack(pady=5)

    tk.Label(root, text="Secure Entrance Gate Account Password", **LABEL_STYLE).pack(anchor="w", padx=100)
    password_entry = tk.Entry(root, width=35, show="*", **ENTRY_STYLE)
    password_entry.pack(pady=5)

    def process_login():
        email = email_entry.get().strip()
        pwd = password_entry.get().strip()

        if not email or not pwd:
            messagebox.showerror("Validation Alert", "Data entry processing fields cannot remain vacant.")
            return

        profile = validate_login(email, pwd)
        if profile:
            if profile["role"] != role:
                messagebox.showerror("Policy Access Violation Exception", f"Security Rejection: Account classification signature maps explicitly to: '{profile['role']}' registry tiers.")
                return
            
            global current_user_id, current_user_name, current_user_role
            current_user_id = str(profile["id"])
            current_user_name = profile["name"]
            current_user_role = profile["role"]

            messagebox.showinfo("Handshake Complete", f"Session successfully initialized. Welcome back, {current_user_name}!")
            
            if current_user_role == "Employer":
                show_employer_dashboard()
            else:
                show_job_seeker_dashboard()
        else:
            messagebox.showerror("Authentication Fault", "Invalid access coordinates matched against database ledger records.")

    submit_btn = tk.Button(root, text="Authenticate Credentials", **BUTTON_STYLE, width=22, command=process_login)
    submit_btn.pack(pady=20)

    reg_lnk = tk.Button(root, text="Don't possess a verified account? Register here", font=("Helvetica", 10, "underline"), bg=BACKGROUND_COLOR, fg=TEXT_COLOR, bd=0, activebackground=BACKGROUND_COLOR, command=lambda: show_register(role))
    reg_lnk.pack(pady=10)

    back_btn = tk.Button(root, text="Return to Role Gateway Selection Panels", **BUTTON_STYLE, bg="#777777", activebackground="#555555", width=25, command=show_role_selection)
    back_btn.pack(pady=15)

def show_register(role):
    clear_screen()
    root.geometry("550x650")

    title = tk.Label(root, text=f"Establish New {role} Identity Node", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=25)

    tk.Label(root, text="Full Legal Title Name / Authorized Entity Branding Name", **LABEL_STYLE).pack(anchor="w", padx=100)
    name_entry = tk.Entry(root, width=35, **ENTRY_STYLE)
    name_entry.pack(pady=5)

    tk.Label(root, text="Primary Communication Email Gateway Handle", **LABEL_STYLE).pack(anchor="w", padx=100)
    email_entry = tk.Entry(root, width=35, **ENTRY_STYLE)
    email_entry.pack(pady=5)

    tk.Label(root, text="Cryptographic Verification Profile Entrance Password", **LABEL_STYLE).pack(anchor="w", padx=100)
    password_entry = tk.Entry(root, width=35, show="*", **ENTRY_STYLE)
    password_entry.pack(pady=5)

    label_txt = "Primary Operations / Market Operational Domain Focus Sector" if role == "Employer" else "Core Professional Functional Competency Trait Specialization"
    tk.Label(root, text=label_txt, **LABEL_STYLE).pack(anchor="w", padx=100)
    
    selected_skill_var = tk.StringVar()
    selected_skill_var.set(SKILL_POOL[0]) 
    
    skill_dropdown = ttk.OptionMenu(root, selected_skill_var, SKILL_POOL[0], *SKILL_POOL)
    skill_dropdown.config(width=32)
    skill_dropdown.pack(pady=5)

    def process_registration():
        name = name_entry.get().strip()
        email = email_entry.get().strip()
        pwd = password_entry.get().strip()
        skill = selected_skill_var.get()

        if not name or not email or not pwd:
            messagebox.showerror("Execution Fault Exception", "All identity profile creation fields must be initialized completely.")
            return

        is_saved = save_user(name, email, pwd, role, skill)
        if is_saved:
            messagebox.showinfo("Registry Appended", "Account data metrics successfully indexed into file structures. Please authenticate via portal log window.")
            show_login(role)
        else:
            messagebox.showerror("Identity Mapping Collision", "An identical email handle reference matching this target lookup index already exists on the database records.")

    submit_btn = tk.Button(root, text="Commit Profile Registry Data Node", **BUTTON_STYLE, width=25, command=process_registration)
    submit_btn.pack(pady=20)

    login_lnk = tk.Button(root, text="Already possess a validated workspace profile? Login here", font=("Helvetica", 10, "underline"), bg=BACKGROUND_COLOR, fg=TEXT_COLOR, bd=0, activebackground=BACKGROUND_COLOR, command=lambda: show_login(role))
    login_lnk.pack(pady=10)

    back_btn = tk.Button(root, text="Return to Role Gateway Selection Panels", **BUTTON_STYLE, bg="#777777", activebackground="#555555", width=25, command=show_role_selection)
    back_btn.pack(pady=15)

def show_employer_dashboard():
    clear_screen()
    root.geometry("780x650")

    title = tk.Label(root, text="Corporate Employer Administrative Command Panel", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=25)

    welcome = tk.Label(root, text=f"Active Dashboard Session Root Context Node Operator: {current_user_name}", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=TEXT_COLOR)
    welcome.pack(pady=10)

    post_btn = tk.Button(root, text="Broadcast / List a New Vacant Job Record", **BUTTON_STYLE, width=35, command=show_post_job_screen)
    post_btn.pack(pady=12)

    view_btn = tk.Button(root, text="Track Published Open Job Vacancies Index", **BUTTON_STYLE, width=35, command=show_employer_jobs_screen)
    view_btn.pack(pady=12)

    logout_btn = tk.Button(root, text="Close Active Identity Session Interface Connection", **BUTTON_STYLE, bg=ERROR_COLOR, activebackground="#D62828", width=35, command=show_role_selection)
    logout_btn.pack(pady=12)

def show_post_job_screen():
    clear_screen()

    title = tk.Label(root, text="Publish Active Vacancy Metrics Record", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=20)

    tk.Label(root, text="Designated Functional Job Opening Title Designation", **LABEL_STYLE).pack(anchor="w", padx=150)
    title_entry = tk.Entry(root, width=50, **ENTRY_STYLE)
    title_entry.pack(pady=5)

    tk.Label(root, text="Operational Domain Daily Role Tasks Overview Description", **LABEL_STYLE).pack(anchor="w", padx=150)
    desc_entry = tk.Text(root, width=50, height=5, font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR, relief="solid", bd=1)
    desc_entry.pack(pady=5)

    tk.Label(root, text="Mandatory Core Domain Knowledge Prerequisite Asset", **LABEL_STYLE).pack(anchor="w", padx=150)
    job_skill_var = tk.StringVar()
    job_skill_var.set(SKILL_POOL[0])
    
    job_skill_dropdown = ttk.OptionMenu(root, job_skill_var, SKILL_POOL[0], *SKILL_POOL)
    job_skill_dropdown.config(width=46)
    job_skill_dropdown.pack(pady=5)

    tk.Label(root, text="Minimum Mandatory Experience Level Tier Requirement", **LABEL_STYLE).pack(anchor="w", padx=150)
    exp_var = tk.StringVar()
    exp_var.set("Entry-Level")
    exp_menu = tk.OptionMenu(root, exp_var, "No Experience", "Entry-Level", "Intermediate", "Expert")
    exp_menu.config(font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR, width=15)
    exp_menu.pack(pady=8)

    def handle_job_publish():
        t_text = title_entry.get().strip()
        d_text = desc_entry.get("1.0", tk.END).strip()
        s_text = job_skill_var.get()
        e_text = exp_var.get()

        if not t_text or not d_text:
            messagebox.showerror("Halted Publishing Request", "Structural role text inputs cannot contain null or whitespace boundaries.")
            return

        save_job(current_user_id, t_text, d_text, s_text, e_text)
        messagebox.showinfo("Database Record Appended", "Vacancy ledger record appended to storage frameworks successfully.")
        show_employer_dashboard()

    save_btn = tk.Button(root, text="Broadcast Entry to Server Matrix", **BUTTON_STYLE, width=25, command=handle_job_publish)
    save_btn.pack(pady=15)

    back_btn = tk.Button(root, text="Abort Operation & Discard Changes", **BUTTON_STYLE, bg="#777777", activebackground="#555555", width=25, command=show_employer_dashboard)
    back_btn.pack()

def show_employer_jobs_screen():
    clear_screen()

    title = tk.Label(root, text="Your Tracked Published Job Opening Matrix", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=15)

    canvas = tk.Canvas(root, bg=BACKGROUND_COLOR, highlightthickness=0)
    scrollbar = tk.Scrollbar(root, orient="vertical", command=canvas.yview)
    scroll_frame = tk.Frame(canvas, bg=BACKGROUND_COLOR)

    scroll_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
    canvas.create_window((0, 0), window=scroll_frame, anchor="nw", width=740)
    canvas.configure(yscrollcommand=scrollbar.set)

    canvas.pack(side="left", fill="both", expand=True, padx=20)
    scrollbar.pack(side="right", fill="y")

    my_jobs = get_employer_jobs(current_user_id)

    if not my_jobs:
        tk.Label(scroll_frame, text="Zero published vacancy records tracked on file archives under this operator credential node.", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT).pack(pady=40)
    else:
        for job in my_jobs:
            j_id = job["id"]
            box = tk.Frame(scroll_frame, bg=BACKGROUND_COLOR, bd=1, relief="solid")
            box.pack(fill="x", padx=10, pady=10, ipady=5)

            tk.Label(box, text=job["title"], font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR).pack(anchor="w", padx=15, pady=5)
            tk.Label(box, text=f"Experience Tier Constraints: {job['experience']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15)
            tk.Label(box, text=f"Prerequisite Domain Skill Element: {job['skills']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15)
            tk.Label(box, text=f"Description Overview Body: {job['description']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=MUTED_TEXT, wraplength=600, justify="left").pack(anchor="w", padx=15, pady=5)

            review_btn = tk.Button(box, text="Review Interested Candidate Applications Pool", **BUTTON_STYLE, command=lambda target_id=j_id, target_title=job["title"]: show_applicants_screen(target_id, target_title))
            review_btn.pack(anchor="e", padx=15, pady=5)

    back_btn = tk.Button(root, text="Return to Corporate Command Hub Console", **BUTTON_STYLE, bg="#777777", activebackground="#555555", command=show_employer_dashboard)
    back_btn.pack(side="bottom", pady=15)

def show_applicants_screen(job_id, job_title):
    clear_screen()

    title = tk.Label(root, text=f"Candidate Ledger Match Matrix Log:\n{job_title}", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=20)

    candidates = get_job_applicants(job_id)

    if not candidates:
        tk.Label(root, text="No candidate submission records logged via matching relational table queries for this tracking node index.", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT).pack(pady=50)
    else:
        for candidate in candidates:
            c_box = tk.Frame(root, bg=BACKGROUND_COLOR, bd=1, relief="solid")
            c_box.pack(fill="x", padx=40, pady=8, ipady=6)

            tk.Label(c_box, text=candidate["name"], font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15, pady=2)
            tk.Label(c_box, text=f"Secure Identity Communication Gateway Handle: {candidate['email']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=MUTED_TEXT).pack(anchor="w", padx=15)
            tk.Label(c_box, text=f"Verified Profile Specialization Core Competencies: {candidate['skills']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15, pady=2)

    back_btn = tk.Button(root, text="Return to Active Vacancies Tracking View", **BUTTON_STYLE, bg="#777777", activebackground="#555555", command=show_employer_jobs_screen)
    back_btn.pack(side="bottom", pady=25)

def show_job_seeker_dashboard():
    """Mounts the interactive subclass panel view layout cleanly inside our application window container frame context."""
    clear_screen()
    
    session_credentials_package = {
        "id": str(current_user_id),
        "name": current_user_name,
        "role": current_user_role
    }
    
    sub_dashboard_frame = tk.Frame(root, bg=BACKGROUND_COLOR)
    sub_dashboard_frame.pack(fill="both", expand=True)
    
    seeker_panel = EmployeeDashboard(master=sub_dashboard_frame, current_user=session_credentials_package)
    seeker_panel.pack(fill="both", expand=True)
    
    logout_bar = tk.Button(root, text="Terminate Profile Authentication Link Connection & Securely Log Out", **BUTTON_STYLE, bg=ERROR_COLOR, activebackground="#D62828", command=show_role_selection)
    logout_bar.pack(side="bottom", fill="x", pady=5)

if __name__ == "__main__":
    show_role_selection()
    root.mainloop()